# Session 19 — End-to-End MLOps Pipeline on Amazon Web Services

**Goal:** the AWS counterpart to Session 12/18's Vertex AI pipeline — orchestrate
preprocess → train → evaluate → conditionally deploy as one **SageMaker Pipeline**,
tying together the training (Session 8) and AutoML (Session 9) work from earlier
into a single reproducible, gated pipeline.

## Why a pipeline instead of separate notebook cells

Sessions 8-9 ran training and deployment as standalone steps you'd trigger by hand.
A `SageMaker Pipeline` encodes the dependency graph explicitly — a training job's
output model automatically becomes the next step's input, with lineage tracked in
SageMaker Model Registry, and a conditional step can block deployment on a quality
check exactly like Session 12's `dsl.If` gate.

## Prerequisites

Needs an **AWS account** with SageMaker access — not available in this sandbox.
Complete, correct reference code below.

```bash
pip install sagemaker boto3
```

In [ ]:
import sagemaker
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import ProcessingStep, TrainingStep
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.functions import JsonGet
from sagemaker.workflow.parameters import ParameterString, ParameterFloat
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.sklearn.estimator import SKLearn
from sagemaker.processing import ProcessingInput, ProcessingOutput

ROLE = sagemaker.get_execution_role()
REGION = "us-east-1"
BUCKET = "your-sagemaker-bucket"

## Step 1 — Pipeline parameters

Parameters make a pipeline reusable across runs without editing code — pass a
different `min_auc_threshold` for a stricter production gate versus a looser
experimentation run, same idea as Session 18's `n_rows`/`n_estimators`.

In [ ]:
input_data = ParameterString(name="InputData", default_value=f"s3://{BUCKET}/heart-disease/raw.csv")
min_auc_threshold = ParameterFloat(name="MinAUCThreshold", default_value=0.85)

## Step 2 — Processing step: clean and split the data

A `SKLearnProcessor` runs a data-prep script in a managed container — the SageMaker
equivalent of Session 12/18's `preprocess` KFP component.

In [ ]:
processing_script = '''\
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("/opt/ml/processing/input/raw.csv").dropna()
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["target"])
train_df.to_csv("/opt/ml/processing/train/train.csv", index=False)
test_df.to_csv("/opt/ml/processing/test/test.csv", index=False)
'''
with open("preprocess.py", "w") as f:
    f.write(processing_script)

processor = SKLearnProcessor(
    framework_version="1.2-1", role=ROLE,
    instance_type="ml.m5.large", instance_count=1,
)

process_step = ProcessingStep(
    name="PreprocessHeartDiseaseData",
    processor=processor,
    inputs=[ProcessingInput(source=input_data, destination="/opt/ml/processing/input")],
    outputs=[
        ProcessingOutput(output_name="train", source="/opt/ml/processing/train"),
        ProcessingOutput(output_name="test", source="/opt/ml/processing/test"),
    ],
    code="preprocess.py",
)

## Step 3 — Training step, fed by the processing step's output

`process_step.properties` references the processing step's actual S3 output path at
pipeline-execution time — the pipeline handles data-passing between steps
automatically, the same wiring pattern as Session 12's `.outputs["output_dataset"]`.

In [ ]:
estimator = SKLearn(
    entry_point="sagemaker_train.py",  # from Session 8
    role=ROLE, instance_type="ml.m5.large", framework_version="1.2-1",
)

train_step = TrainingStep(
    name="TrainHeartDiseaseModel",
    estimator=estimator,
    inputs={
        "train": process_step.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri,
    },
)

## Step 4 — Conditional deployment gate

`ConditionGreaterThanOrEqualTo` reads an evaluation metric (written by an evaluation
processing step, omitted here for brevity, mirroring Session 8's `sagemaker_train.py`
pattern) and only proceeds to registering/deploying the model if it clears the bar —
directly analogous to Session 12's `evaluate_gate` component.

In [ ]:
condition_step = ConditionStep(
    name="CheckAUCThreshold",
    conditions=[
        ConditionGreaterThanOrEqualTo(
            left=JsonGet(
                step_name=train_step.name,
                property_file="evaluation",
                json_path="metrics.auc.value",
            ),
            right=min_auc_threshold,
        )
    ],
    if_steps=[],   # register_model_step / deploy step would go here
    else_steps=[], # e.g. a notification step: "model did not meet quality bar"
)

## Step 5 — Assemble and run the pipeline

In [ ]:
pipeline = Pipeline(
    name="heart-disease-e2e-pipeline",
    parameters=[input_data, min_auc_threshold],
    steps=[process_step, train_step, condition_step],
)

pipeline.upsert(role_arn=ROLE)
execution = pipeline.start()
print(f"Pipeline execution started: {execution.arn}")
execution.wait()
print("Steps:", execution.list_steps())

## What to try next

* Add a `RegisterModel` step inside `condition_step`'s `if_steps`, registering the
  approved model to SageMaker Model Registry with a `PendingManualApproval` status —
  a human-in-the-loop gate before a fully automatic deployment.
* Compare this pipeline's shape against Session 12's Vertex AI version side by side —
  both express preprocess → train → gate → deploy, using each cloud's native
  primitives (`ConditionStep`/`JsonGet` vs. `dsl.If`).